In [12]:
from utils.paths import RAW_DIR,MODELS_DIR
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import pickle

In [13]:
train_ann = pd.read_csv(RAW_DIR/"Train_ERANN.csv")
val_ann = pd.read_csv(RAW_DIR/"Val_ERANN.csv")

total_ann = pd.concat([train_ann, val_ann], axis=0).reset_index(drop=True)
total_ann = total_ann[["r","del_r","del_EN", "S", "VEC"]]

scaler = MinMaxScaler()
scaler.fit_transform(total_ann)

with open(MODELS_DIR / "scaler_ANN_customized.pkl", "wb") as f:
    pickle.dump(scaler, f)

In [2]:
import requests
import pandas as pd
import numpy as np
from io import StringIO

# ====================== Valence Electron ================================
class ElementTableProp:
    def __init__(self, url, table_index=0, timeout=30):
        self.url = url
        self.table_index = table_index
        self.timeout = timeout

    def read_table(self):
        r = requests.get(self.url, headers={"User-Agent": "Mozilla/5.0"}, timeout=self.timeout)
        r.raise_for_status()
        tables = pd.read_html(StringIO(r.text))
        if self.table_index >= len(tables):
            raise IndexError(f"Requested table {self.table_index}, but only {len(tables)} tables found.")
        return tables[self.table_index].reset_index(drop=True)

    @staticmethod
    def _find_symbol_col(df):
        for c in df.columns:
            if str(c).strip().lower() == "symbol":
                return c
        raise KeyError(f"No Symbol/symbol column found. Columns: {list(df.columns)}")

    def extract_property(
        self,
        value_col,
        out_value_name=None,
        dropna=True,
        numeric=True,
        extract_digits=False,
    ):
        df = self.read_table()

        sym_col = self._find_symbol_col(df)

        out_value_name = out_value_name or value_col

        out = df[[sym_col, value_col]].copy()
        out = out.rename(columns={sym_col: "symbol", value_col: out_value_name})

        if extract_digits:
            out[out_value_name] = (
                out[out_value_name].astype(str).str.extract(r"([-+]?\d*\.?\d+)")[0]
            )

        if numeric:
            out[out_value_name] = pd.to_numeric(out[out_value_name], errors="coerce")

        if dropna:
            out = out.dropna(subset=[out_value_name])

        out["symbol"] = out["symbol"].astype(str).str.strip()
        out = out.drop_duplicates(subset=["symbol"]).reset_index(drop=True)

        return out
    @staticmethod
    def _last_numeric(row):
        nums = pd.to_numeric(row, errors="coerce").dropna()
        return nums.iloc[-1] if len(nums) else np.nan

    def extract_valence_electrons(self, legend_col="Legend", name_row_start=1, row_step=3, valence_row_offset=3):
        df = self.read_table()

        if legend_col not in df.columns:
            raise KeyError(f"Column '{legend_col}' not found. Columns: {list(df.columns)}")

        df_name = df.iloc[name_row_start::row_step].copy()
        df_val = df.iloc[valence_row_offset::row_step].copy()

        df_name["symbol"] = df_name[legend_col].astype(str).str.extract(r"\b\d+\s+([A-Z][a-z]?)\b")
        df_name = df_name.reset_index(drop=True)

        df_val["valence"] = df_val.apply(self._last_numeric, axis=1)
        df_val = df_val.reset_index(drop=True)

        out = pd.DataFrame({"symbol": df_name["symbol"], "valence": df_val["valence"]})
        out = out.dropna(subset=["symbol"]).drop_duplicates(subset=["symbol"]).reset_index(drop=True)
        return out
    
    @staticmethod
    def to_map(df, value_col):
        return dict(zip(df["symbol"], df[value_col]))

Atomic Radius

In [5]:
url_atr = "https://en.wikipedia.org/wiki/Atomic_radii_of_the_elements_(data_page)"
wiki_atr = ElementTableProp(url_atr, table_index=0)

df_atomic_radius = wiki_atr.extract_property(
    value_col="Metallic",
    out_value_name="atomic_radius_metallic",
    numeric=True,
    extract_digits=True,   # pulls number out of strings like "145 pm"
    dropna=True
)

df_atomic_radius.head()
atomic_radius_map = ElementTableProp.to_map(df_atomic_radius, "atomic_radius_metallic") # units pm


Electronegativity difference

In [6]:
url_electron = "https://en.wikipedia.org/wiki/Electronegativities_of_the_elements_(data_page)"
wiki_en = ElementTableProp(url_electron, table_index=1)

df_electronegativity = wiki_en.extract_property(
    value_col="electronegativity",
    out_value_name="electronegativity",
    numeric=True,
    extract_digits=False,
    dropna=True
)

df_electronegativity.head()
en_map = ElementTableProp.to_map(df_electronegativity, "electronegativity")

Valence Electron

In [21]:
url_valence = "https://en.wikipedia.org/wiki/Electron_configurations_of_the_elements_(data_page)"
wiki_val = ElementTableProp(url_valence, table_index=0)  # change index if needed

df_valence = wiki_val.extract_valence_electrons(legend_col="Legend")
df_valence.loc[df_valence['symbol']=='Ni','valence'] = 10
df_valence.loc[df_valence['symbol']=='Cu','valence'] = 11

df_valence.head()
df_valence.to_pickle(DATA_DIR/"descriptors_data/Valence_electrons.pkl")

Matminer (Information)

In [6]:
from mp_api.client import MPRester
import pandas as pd
import numpy as np

In [4]:
with MPRester("2EEDlkvr2m8hB8XsKaA2lBGV9Ta57jFV") as mpr:
    docs = mpr.materials.summary.search(
        formula="Cu50Ni50",
        fields=["material_id", "formula_pretty", "band_gap", "is_stable"]
    )

df_mp = pd.DataFrame([d.model_dump() for d in docs])
df_mp.head()

Retrieving SummaryDoc documents: 100%|██████████| 3/3 [00:00<00:00, 28597.53it/s]


,formula_pretty,material_id,is_stable,band_gap,fields_not_requested
0,CuNi,mp-crtdv,False,0.0,"[builder_meta, nsites, elements, nelements, co..."
1,CuNi,mp-cpjpd,False,0.0,"[builder_meta, nsites, elements, nelements, co..."
2,CuNi,mp-crted,False,0.0,"[builder_meta, nsites, elements, nelements, co..."


In [7]:
from pymatgen.core import Element

for el in ["Cu", "Ni", "Al"]:
    e = Element(el)
    print(el)
    print("Z:", e.Z)
    print("electronic structure:", e.electronic_structure)
    print()

Cu
Z: 29
electronic structure: [Ar].3d10.4s1

Ni
Z: 28
electronic structure: [Ar].3d8.4s2

Al
Z: 13
electronic structure: [Ne].3s2.3p1



In [ ]:
import pandas as pd
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty

# create dataframe
df = pd.DataFrame({"composition": [Composition("LiFePO4")]})

# load featurizer
featurizer = ElementProperty.from_preset("magpie")

# featurize
df_feat = featurizer.featurize_dataframe(df, "composition")

print(df_feat.head())

In [15]:

from mendeleev import element

cu = element("Cu")
ni = element("Ni")
al = element("Al")

print("Allen EN:")
print("Cu:", cu.en_allen)
print("Ni:", ni.en_allen)
print("Al:", al.en_allen)

print("\nPauling EN:")
print("Cu:", cu.en_pauling)
print("Ni:", ni.en_pauling)
print("Al:", al.en_pauling)

print("\nMetalic radius EN:")
print("Cu:", cu.atomic_radius_rahm)
print("Ni:", ni.atomic_radius_rahm)
print("Al:", al.atomic_radius_rahm)

print("\nIonenergies")
print("Cu:", cu.ionenergies)
print("Ni:", ni.ionenergies)
print("Al:", al.ionenergies)

print("\nelectron_affinity")
print("Cu:", cu.electron_affinity)
print("Ni:", ni.electron_affinity)
print("Al:", al.electron_affinity)


Allen EN:
Cu: 10.96
Ni: 11.13
Al: 9.539

Pauling EN:
Cu: 1.9
Ni: 1.91
Al: 1.61

Metalic radius EN:
Cu: 217.0
Ni: 229.0
Al: 239.0

Ionenergies
Cu: {1: 7.72638, 2: 20.29239, 3: 36.841, 4: 57.38, 5: 79.8, 6: 103.0, 7: 139.0, 8: 166.0, 9: 198.0, 10: 232.2, 11: 265.33, 12: 367.0, 13: 401.0, 14: 436.0, 15: 483.1, 16: 518.7, 17: 552.8, 18: 632.5, 19: 670.608, 20: 1690.5, 21: 1800.0, 22: 1918.0, 23: 2044.0, 24: 2179.4, 25: 2307.3, 26: 2479.1, 27: 2586.954, 28: 11062.4309, 29: 11567.6237}
Ni: {1: 7.639878, 2: 18.168838, 3: 35.187, 4: 54.92, 5: 76.06, 6: 108.0, 7: 132.0, 8: 162.0, 9: 193.2, 10: 224.7, 11: 319.5, 12: 351.6, 13: 384.5, 14: 429.3, 15: 462.8, 16: 495.4, 17: 571.07, 18: 607.02, 19: 1540.1, 20: 1646.0, 21: 1758.0, 22: 1880.0, 23: 2008.1, 24: 2130.5, 25: 2295.6, 26: 2399.259, 27: 10288.8848, 28: 10775.3948}
Al: {1: 5.985769, 2: 18.82855, 3: 28.447642, 4: 119.9924, 5: 153.8252, 6: 190.49, 7: 241.76, 8: 284.64, 9: 330.21, 10: 398.65, 11: 442.005, 12: 2085.97693, 13: 2304.140359}

electro